# Diabetes Risk Studio — EDA, leakage-safe modeling, and explainability

**Classroom walkthrough.** This notebook explores the included Pima Indians dataset, runs the reproducible training pipeline, then inspects its held-out evaluation and SHAP reports. This is an educational demo, not a diagnostic tool.

> Run from this notebook with the project environment installed. The modeling cell runs four classifiers and may take a few minutes.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
data_path = ROOT / 'data' / 'diabetes.csv'
df = pd.read_csv(data_path)
print(f'{len(df)} records · {df.shape[1]-1} input features')
display(df.head())

## 1. Inspect the data

The target `Outcome` is binary. Several clinical columns contain zeros that represent missing measurements in this dataset; they are not treated as true physiological zeroes.

In [ ]:
display(df.describe().T.round(2))
print('Outcome proportions:')
display(df['Outcome'].value_counts(normalize=True).rename('proportion').to_frame())
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
zero_rates = (df[zero_cols].eq(0).mean() * 100).sort_values(ascending=False)
display(zero_rates.rename('zero values (%)').round(1).to_frame())

In [ ]:
ax = df['Outcome'].value_counts().sort_index().plot(kind='bar', color=['#4C9F70','#D95D5D'], rot=0)
ax.set(title='Target class counts', xlabel='Outcome (0 = no diabetes, 1 = diabetes)', ylabel='Patients')
plt.tight_layout()
plt.show()

## 2. Train and compare models

The training script uses a stratified 80/20 split. Median imputation, scaling, and SMOTE are steps inside an imbalanced-learn pipeline, so cross-validation refits them within each training fold. The untouched test split is used only for final metrics. Model selection uses mean training-fold ROC-AUC.

The four models are Logistic Regression, Random Forest, AdaBoost, and Stacking. To reproduce all saved artifacts, run:

In [ ]:
import runpy
runpy.run_path(str(ROOT / 'train_models.py'), run_name='__main__')

## 3. Review the held-out results

Accuracy alone can hide poor minority-class performance. Compare precision with recall, and look at balanced accuracy, specificity, and ROC-AUC together. The Brier score is a probability error measure (lower is better); it is not a calibration guarantee.

In [ ]:
metrics = pd.read_csv(ROOT / 'reports' / 'model_comparison.csv')
display(metrics.style.format(precision=3))
display(Image(filename=str(ROOT / 'reports' / 'model_curves.png')))

## 4. Explainability

The global SHAP summary describes which features the Random Forest uses and how their values move its output. These are model associations, not causal or clinical explanations. The app also shows a local SHAP waterfall for a single example profile.

In [ ]:
display(Image(filename=str(ROOT / 'reports' / 'shap_summary.png')))
display(Image(filename=str(ROOT / 'reports' / 'shap_feature_importance.png')))

## 5. Limitations and discussion

- The dataset is small and represents Pima Indian women; it cannot establish performance for other populations.
- One holdout split has statistical uncertainty. Stronger evidence would require repeated or nested validation and independent external data.
- SMOTE changes the training distribution; threshold choice and calibration deserve separate analysis.
- SHAP explains model behavior but does not identify causes. The app's score bands are illustrative, not medical thresholds.
- This project is for education and must not be used for diagnosis or treatment decisions.

**Discussion prompt:** Which mistake would be more costly in a screening workflow, a false negative or a false positive, and how would that affect a threshold decision? The answer depends on the real clinical context and cannot be settled by maximizing accuracy alone.